# Z-Image LoRA Training

Run cells in order. Cell 1 starts training in background (non-blocking).
Cell 2 monitors progress. Cell 3 downloads the result.

In [ ]:
# Cell 1: Start training in background
import subprocess, os

log_path = '/workspace/train.log'
log = open(log_path, 'w')
p = subprocess.Popen(
    ['python3', '/workspace/train_lora.py'],
    stdout=log, stderr=subprocess.STDOUT,
    cwd='/workspace'
)
print(f'Training PID: {p.pid}')
print(f'Log: {log_path}')
print(f'Monitor with: !tail -f {log_path}')

In [ ]:
# Cell 2: Check progress
import os, time

log = '/workspace/train.log'
if os.path.exists(log):
    # Show last 30 lines
    with open(log) as f:
        lines = f.readlines()[-30:]
    print(''.join(lines))
else:
    print('No log yet')

# Check GPU
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'], capture_output=True, text=True)
print(f'GPU: {r.stdout.strip()}')

# Check output
out_dir = '/workspace/lora_output'
if os.path.exists(out_dir):
    for d in os.listdir(out_dir):
        path = os.path.join(out_dir, d, 'lora_weights.pt')
        if os.path.exists(path):
            print(f'Checkpoint: {path} ({os.path.getsize(path)/1e6:.1f}MB)')

In [ ]:
# Cell 3: Download LoRA result
# Run this only after training is complete
import os, subprocess

final = '/workspace/lora_output/lora_final/lora_weights.pt'
if os.path.exists(final):
    size = os.path.getsize(final) / 1e6
    print(f'Final LoRA: {final} ({size:.1f}MB)')
    # Upload to catbox for download
    r = subprocess.run(['curl', '-s', '-F', f'fileToUpload=@{final}', 'https://catbox.moe/user/api.php?reqtype=fileupload'], capture_output=True, text=True)
    print(f'Download URL: {r.stdout}')
else:
    print('Training not finished yet. Check Cell 2 for progress.')